# Experiment 56 - Exp22 Logit Temperature Calibration

**Building on:** exp53 (OOF cmAP=0.94973) and exp55 rank-blend failure.

**Hypothesis:** Rank blending was too destructive, but high-lambda prior logits may still be overconfident. Test a gentler calibration: scale the final blended logits by a global temperature before the standard sigmoid/postprocessing path.


In [ ]:
import json
from pathlib import Path

nb_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
exp49_path = nb_dir / 'exp49_exp22_public_aware.ipynb'
exp49_nb = json.loads(exp49_path.read_text())

for cell_idx in range(1, 10):
    print(f'--- executing exp49 cell {cell_idx} ---')
    src = ''.join(exp49_nb['cells'][cell_idx]['source'])
    exec(compile(src, f'exp49_cell_{cell_idx}', 'exec'), globals())


In [ ]:
# Exp56: global logit temperature around exp53 plateau

LAMBDAS_EXT = [4.0, 5.0]
TTA_W0S = [0.85, 0.90]
POWERS_EXT = [1.0, 1.1, 1.2]
ALPHAS = [0.05, 0.08, 0.10]
TEMPS = [0.75, 0.90, 1.00, 1.10, 1.25, 1.50]
ENSEMBLE_GRIDS = [
    (0.02, 0.58, 0.40),
    (0.02, 0.63, 0.35),
    (0.05, 0.55, 0.40),
]

def postprocess_temp(logits, temp=1.0, power=0.05, base_alpha=0.10):
    p = sigmoid((logits / temp) / temperatures[None, :])
    p = file_confidence_scale(p, power=power)
    p = adaptive_delta_smooth(p, base_alpha=base_alpha)
    return np.clip(p, 0.0, 1.0)

best_cmap = 0.0
best_cfg = {}
results = []
prior_cache = {}

def get_prior_pair_exp56(lam):
    if lam not in prior_cache:
        prior_cache[lam] = (
            apply_prior(oof_raw_0, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
            apply_prior(oof_raw_250, oof_meta_sites, oof_meta_hours, prior_tables, lam,
                        prior_shrink=1.0, prior_logit_clip=None),
        )
    return prior_cache[lam]

for lam in LAMBDAS_EXT:
    prior_0, prior_250 = get_prior_pair_exp56(lam)
    for wp, wm, wpr in ENSEMBLE_GRIDS:
        fp_0 = wp * oof_proto_0 + wm * oof_mlp_0 + wpr * prior_0
        fp_250 = wp * oof_proto_250 + wm * oof_mlp_250 + wpr * prior_250
        for tta_w0 in TTA_W0S:
            fp_avg = tta_w0 * fp_0 + (1.0 - tta_w0) * fp_250
            for temp in TEMPS:
                for power in POWERS_EXT:
                    for alpha in ALPHAS:
                        probs = postprocess_temp(fp_avg, temp=temp, power=power, base_alpha=alpha)
                        cmap = padded_cmap(Y_FULL_aligned, probs)
                        row = {'lam': lam, 'tta_w0': tta_w0, 'temp': temp, 'power': power, 'alpha': alpha,
                               'wp': wp, 'wm': wm, 'wpr': wpr, 'cmap': cmap}
                        results.append(row)
                        if cmap > best_cmap:
                            best_cmap = cmap
                            best_cfg = row.copy()

df = pd.DataFrame(results).sort_values('cmap', ascending=False).reset_index(drop=True)
print(f'Best OOF cmAP: {best_cmap:.5f}')
print(f'Best config:   {best_cfg}')
print('\nTop 25 configs:')
print(df.head(25).to_string(index=False))
print('\nBest cmAP by temp:')
print(df.groupby('temp')['cmap'].max().reset_index().to_string(index=False))

prior_0_f, prior_250_f = get_prior_pair_exp56(best_cfg['lam'])
fp_0_f = best_cfg['wp'] * oof_proto_0 + best_cfg['wm'] * oof_mlp_0 + best_cfg['wpr'] * prior_0_f
fp_250_f = best_cfg['wp'] * oof_proto_250 + best_cfg['wm'] * oof_mlp_250 + best_cfg['wpr'] * prior_250_f
fp_f = best_cfg['tta_w0'] * fp_0_f + (1.0 - best_cfg['tta_w0']) * fp_250_f
p_best = postprocess_temp(fp_f, temp=best_cfg['temp'], power=best_cfg['power'], base_alpha=best_cfg['alpha'])
auc_b = macro_auc(Y_FULL_aligned, p_best)
cmap_b = padded_cmap(Y_FULL_aligned, p_best)

print('=' * 60)
print('Experiment 56 - Exp22 logit temperature calibration')
print(f"Best: lambda={best_cfg['lam']}, tta_w0={best_cfg['tta_w0']}, temp={best_cfg['temp']}, power={best_cfg['power']}, alpha={best_cfg['alpha']}")
print(f"Weights: wp={best_cfg['wp']}, wm={best_cfg['wm']}, wpr={best_cfg['wpr']}")
print(f'AUC={auc_b:.5f}  cmAP={cmap_b:.5f}')
print(f'vs exp53 OOF (0.94973): delta cmAP = {cmap_b - 0.94973:+.5f}')
print(f'vs exp22 OOF (0.93894): delta cmAP = {cmap_b - 0.93894:+.5f}')
print(f'Total wall time: {(time.time() - _WALL_START)/60:.1f} min')
